# Module 3 • Classical Natural Language Processing

# Lesson 12 • Tokenization and Sentence Segmentation

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Beginner  
**Estimated study time:** 90–120 minutes

---

## Scope

This lesson explains how raw text is divided into sentences and tokens. It covers
whitespace and regular-expression tokenization, punctuation, contractions, structured
tokens, offsets, sentence-boundary ambiguity, multilingual tokenization, Arabic
clitics, subword units, and evaluation.

## Learning Objectives

After completing this lesson, the learner should be able to:

- define tokenization and sentence segmentation;
- explain why token boundaries depend on language and task;
- compare whitespace, rule-based, and regex tokenizers;
- handle punctuation, contractions, numbers, URLs, emails, and hyphens;
- identify common sentence-boundary ambiguities;
- preserve token offsets;
- discuss Arabic clitic segmentation;
- distinguish word, character, and subword tokenization;
- evaluate token and sentence boundaries with precision, recall, and F1.

## Table of Contents

1. What Is Tokenization?
2. Task-Dependent Token Boundaries
3. Whitespace Tokenization
4. Regex and Rule-Based Tokenization
5. Punctuation, Contractions, and Hyphens
6. Numbers, URLs, Emails, and Dates
7. Token Offsets
8. Sentence Segmentation
9. Sentence-Boundary Ambiguity
10. Multilingual Tokenization
11. Arabic Tokenization
12. Character and Subword Units
13. Configurable Pipeline
14. Evaluation and Error Analysis
15. Knowledge Check
16. Exercises
17. Summary and Next Lesson

# 1. What Is Tokenization?

**Tokenization** divides text into units called **tokens**.

```text
Natural language processing is useful.
```

One possible token sequence is:

```text
["Natural", "language", "processing", "is", "useful", "."]
```

**Sentence segmentation** identifies where sentences begin and end.

Tokens may be words, punctuation marks, characters, morphemes, subwords, bytes,
or application-specific symbols. A token is therefore a processing decision rather
than a universal linguistic fact.

In [ ]:
text = "Natural language processing is useful."

print(text.split())
print("\nWhitespace splitting leaves final punctuation attached.")

# 2. Task-Dependent Token Boundaries

Consider:

```text
New York-based company
```

Possible tokenizations include:

```text
["New", "York-based", "company"]
["New", "York", "-", "based", "company"]
["New York", "-", "based", "company"]
```

The preferred policy depends on the downstream task and annotation scheme.

In [ ]:
import pandas as pd

task_policies = pd.DataFrame(
    [
        ("Search", "may normalize punctuation or preserve phrases"),
        ("Named Entity Recognition", "must preserve exact spans"),
        ("Machine Translation", "often uses subword units"),
        ("Morphological analysis", "may split clitics and affixes"),
        ("Sentiment analysis", "may retain emojis and repeated punctuation"),
    ],
    columns=["Task", "Tokenization concern"],
)

task_policies

> **Key Idea**
>
> Tokenization must remain consistent across training, validation, testing, and
> production.

# 3. Whitespace Tokenization

Whitespace tokenization is a useful baseline:

```python
text.split()
```

It is simple, but it does not separate punctuation and may fail for writing systems
that do not consistently mark word boundaries with spaces.

In [ ]:
examples = [
    "The model works.",
    "Hello, world!",
    "Email: sara@example.com",
    "The price is $25.50",
]

for example in examples:
    print(example)
    print(example.split())
    print()

# 4. Regex and Rule-Based Tokenization

A rule-based tokenizer can preserve structured forms while separating punctuation.

Example policy:

- preserve URLs and email addresses;
- preserve dates and decimal numbers;
- preserve apostrophes and hyphens inside words;
- separate remaining punctuation.

In [ ]:
import re

TOKEN_PATTERN = re.compile(
    r"https?://[^\s]+"
    r"|www\.[^\s]+"
    r"|[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}"
    r"|\d{4}-\d{2}-\d{2}"
    r"|[$€£]\d+(?:,\d{3})*(?:\.\d+)?"
    r"|\d+(?:[.,]\d+)*"
    r"|[\w]+(?:[-'’][\w]+)*"
    r"|[^\w\s]",
    flags=re.UNICODE,
)

def regex_tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text)

sample = "Email sara@example.com; the price is $25.50!"
print(regex_tokenize(sample))

Specific patterns should appear before general patterns. Otherwise, a URL, email
address, or decimal number may be split into smaller fragments.

# 5. Punctuation, Contractions, and Hyphens

Punctuation may be attached, separated, or grouped.

```text
Really?!
```

Possible outputs include:

```text
["Really", "?", "!"]
["Really", "?!"]
["Really?!"]
```

The selected policy should reflect the task.

In [ ]:
samples = [
    "Really?!",
    "don't",
    "we're",
    "Sara's",
    "state-of-the-art",
    "Arabic-English",
]

for sample in samples:
    print(f"{sample:<20} -> {regex_tokenize(sample)}")

Contractions create additional choices:

```text
don't → ["don't"]
don't → ["do", "n't"]
don't → ["do", "not"]
```

Apostrophes may also mark possession or appear in names, so one contraction rule does
not cover every case.

# 6. Numbers, URLs, Emails, and Dates

Structured expressions should be recognized before broad punctuation splitting.

Examples:

```text
3.5
2026-07-27
$1,250.50
sara@example.com
https://example.com/help
```

In [ ]:
structured_examples = [
    "Version 3.2",
    "2026-07-27",
    "$1,250.50",
    "sara@example.com",
    "https://example.com/help",
]

for item in structured_examples:
    print(f"{item:<32} -> {regex_tokenize(item)}")

A tokenizer may preserve these expressions or later replace them with placeholders.
Recognition should occur before replacement so boundaries remain correct.

# 7. Token Offsets

Token offsets identify the location of each token in the original string.

They support:

- Named Entity Recognition;
- annotation tools;
- highlighting;
- span-level evaluation;
- alignment with raw text;
- error analysis.

In [ ]:
def tokenize_with_offsets(text: str):
    return [
        {
            "token": match.group(0),
            "start": match.start(),
            "end": match.end(),
        }
        for match in TOKEN_PATTERN.finditer(text)
    ]

offset_text = "Mona visited Cairo."
offset_tokens = tokenize_with_offsets(offset_text)

pd.DataFrame(offset_tokens)

In [ ]:
for item in offset_tokens:
    recovered = offset_text[item["start"]:item["end"]]
    print(item["token"], "==", recovered)

Offset conventions must be documented. Systems may use characters, bytes, inclusive
end positions, or exclusive end positions.

# 8. Sentence Segmentation

A simple segmenter may split after `.`, `?`, or `!` followed by whitespace.

In [ ]:
SENTENCE_BOUNDARY_PATTERN = re.compile(r"(?<=[.!?])\s+")

def simple_sentence_split(text: str) -> list[str]:
    return [
        sentence.strip()
        for sentence in SENTENCE_BOUNDARY_PATTERN.split(text)
        if sentence.strip()
    ]

paragraph = (
    "The model finished. The report was saved! "
    "Did the evaluation pass? Yes."
)

simple_sentence_split(paragraph)

This baseline is useful for controlled text but fails on abbreviations, initials,
decimals, domain names, and some quotation patterns.

# 9. Sentence-Boundary Ambiguity

A period does not always mark the end of a sentence.

```text
Dr. Ahmed arrived.
The score was 3.5.
A. Smith wrote the report.
Visit example.com.
```

In [ ]:
ambiguous_examples = [
    "Dr. Ahmed arrived. He presented the paper.",
    "The score was 3.5. It improved.",
    "A. Smith wrote the report. It was accepted.",
]

for example in ambiguous_examples:
    print(example)
    print(simple_sentence_split(example))
    print()

## 9.1 Abbreviation-Aware Baseline

In [ ]:
ABBREVIATIONS = {
    "Dr.": "Dr<PERIOD>",
    "Mr.": "Mr<PERIOD>",
    "Mrs.": "Mrs<PERIOD>",
    "Prof.": "Prof<PERIOD>",
    "e.g.": "e<PERIOD>g<PERIOD>",
    "i.e.": "i<PERIOD>e<PERIOD>",
}

def abbreviation_aware_sentence_split(text: str) -> list[str]:
    protected = text

    for original, replacement in ABBREVIATIONS.items():
        protected = protected.replace(original, replacement)

    sentences = simple_sentence_split(protected)

    return [
        sentence.replace("<PERIOD>", ".")
        for sentence in sentences
    ]

example = "Dr. Ahmed arrived. He presented the paper."
print(abbreviation_aware_sentence_split(example))

A robust segmenter may combine abbreviation lists, language-specific rules, and
statistical or neural models.

# 10. Multilingual Tokenization

Tokenization varies according to:

- writing system;
- use of spaces;
- punctuation conventions;
- morphology;
- clitics;
- compounding;
- code-switching;
- normalization policy.

In [ ]:
multilingual_examples = pd.DataFrame(
    [
        ("English", "Natural language processing"),
        ("Arabic", "معالجة اللغة الطبيعية"),
        ("Chinese", "自然语言处理"),
        ("French", "l'intelligence artificielle"),
    ],
    columns=["Language", "Example"],
)

multilingual_examples

Whitespace splitting cannot identify internal clitics and is not sufficient for
languages that do not consistently place spaces between words.

# 11. Arabic Tokenization

Arabic orthographic words may contain attached conjunctions, prepositions, definite
articles, future markers, and pronouns.

```text
والكتاب
```

Possible tokenizations:

```text
["والكتاب"]
["و", "الكتاب"]
["و", "ال", "كتاب"]
```

In [ ]:
arabic_examples = pd.DataFrame(
    [
        ("والكتاب", ["و", "ال", "كتاب"]),
        ("بالمدرسة", ["ب", "ال", "مدرسة"]),
        ("كتابها", ["كتاب", "ها"]),
        ("وسيكتبون", ["و", "س", "يكتب", "ون"]),
    ],
    columns=["Surface form", "Illustrative segmentation"],
)

arabic_examples

In [ ]:
ARABIC_PREFIXES = ["و", "ف", "ب", "ك", "ل", "س"]
ARABIC_SUFFIXES = ["ها", "هم", "هن", "كما", "كم", "كن", "نا"]

def educational_arabic_segment(word: str) -> list[str]:
    segments = []
    remaining = word

    if len(remaining) > 3 and remaining[0] in ARABIC_PREFIXES:
        segments.append(remaining[0])
        remaining = remaining[1:]

    if remaining.startswith("ال") and len(remaining) > 3:
        segments.append("ال")
        remaining = remaining[2:]

    matched_suffix = None
    for suffix in sorted(ARABIC_SUFFIXES, key=len, reverse=True):
        if remaining.endswith(suffix) and len(remaining) > len(suffix) + 1:
            matched_suffix = suffix
            remaining = remaining[:-len(suffix)]
            break

    if remaining:
        segments.append(remaining)

    if matched_suffix:
        segments.append(matched_suffix)

    return segments

for word in ["والكتاب", "بالمدرسة", "كتابها"]:
    print(word, "->", educational_arabic_segment(word))

This function is educational, not a production Arabic segmenter. Surface rules can
over-segment lexical letters that resemble clitics. Context and morphology are often
required.

# 12. Character and Subword Units

Word tokenization may produce a large vocabulary and unknown words.

Alternatives include:

- character units;
- byte units;
- subword units;
- morpheme-aware units.

In [ ]:
word = "tokenization"

print("Word token:", [word])
print("Character tokens:", list(word))

Common subword approaches include:

- Byte Pair Encoding;
- WordPiece;
- Unigram Language Model;
- SentencePiece;
- byte-level tokenization.

Illustrative segmentation:

```text
tokenization → token + ization
unhappiness  → un + happiness
```

Actual segmentation depends on the learned tokenizer vocabulary.

# 13. Configurable Tokenization Pipeline

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class TokenizationConfig:
    lowercase: bool = False
    return_offsets: bool = False

def configurable_tokenize(
    text: str,
    config: TokenizationConfig = TokenizationConfig(),
):
    working_text = text.lower() if config.lowercase else text

    if config.return_offsets:
        return tokenize_with_offsets(working_text)

    return regex_tokenize(working_text)

In [ ]:
sample = "A State-of-the-Art model costs $25.50."

for config in [
    TokenizationConfig(),
    TokenizationConfig(lowercase=True),
    TokenizationConfig(return_offsets=True),
]:
    print(config)
    print(configurable_tokenize(sample, config))
    print()

The tokenizer configuration should be versioned together with preprocessing and model
artifacts.

# 14. Evaluation and Error Analysis

Tokenization can be evaluated using predicted and reference boundaries.

Metrics include:

- boundary precision;
- boundary recall;
- boundary F1;
- exact sentence segmentation;
- token-level accuracy;
- downstream task performance.

In [ ]:
def boundaries_from_tokens(tokens: list[str]) -> set[int]:
    boundaries = set()
    position = 0

    for token in tokens[:-1]:
        position += len(token)
        boundaries.add(position)

    return boundaries

gold_tokens = ["New", "York-based", "company"]
predicted_tokens = ["New", "York", "-", "based", "company"]

gold_boundaries = boundaries_from_tokens(gold_tokens)
predicted_boundaries = boundaries_from_tokens(predicted_tokens)

print("Gold boundaries:", gold_boundaries)
print("Predicted boundaries:", predicted_boundaries)

In [ ]:
true_positive = len(gold_boundaries & predicted_boundaries)

precision = true_positive / len(predicted_boundaries)
recall = true_positive / len(gold_boundaries)
f1 = 2 * precision * recall / (precision + recall)

print(f"Boundary precision: {precision:.3f}")
print(f"Boundary recall:    {recall:.3f}")
print(f"Boundary F1:        {f1:.3f}")

The simplified boundary example ignores spaces. Production evaluation should compare
offsets in the original text or use a benchmark's official scoring method.

## 14.1 Common Errors

- punctuation attached incorrectly;
- URLs or emails split into fragments;
- decimal numbers split at the period;
- dates handled inconsistently;
- contractions or possessives mishandled;
- hyphenated words over-segmented;
- sentence breaks inserted after abbreviations;
- Arabic clitics ignored or over-segmented;
- offsets misaligned after normalization;
- different tokenizers used during training and deployment.

In [ ]:
error_examples = pd.DataFrame(
    [
        ("3.5", ["3", ".", "5"], "decimal split"),
        ("sara@example.com", ["sara", "@", "example", ".", "com"], "email split"),
        ("Dr. Ahmed", ["Dr", "."], "false sentence boundary"),
        ("والكتاب", ["والكتاب"], "clitics not segmented"),
        ("state-of-the-art", ["state", "-", "of", "-", "the", "-", "art"], "policy-dependent"),
    ],
    columns=["Input", "Possible output", "Issue"],
)

error_examples

# 15. Knowledge Check

1. What is tokenization?
2. Why are token boundaries task-dependent?
3. What are the limitations of whitespace splitting?
4. Why should specific regex patterns precede general patterns?
5. Why are contractions difficult to tokenize?
6. What are token offsets used for?
7. Why does a period not always mark a sentence boundary?
8. How does multilingual tokenization differ across writing systems?
9. What are Arabic clitics?
10. How do word, character, and subword units differ?
11. What does boundary precision measure?
12. Why should downstream performance also be evaluated?

# 16. Exercises

## Exercise 1 — Whitespace Baseline

Apply whitespace tokenization to twenty sentences and categorize the errors.

## Exercise 2 — Regex Tokenizer

Extend the tokenizer to recognize hashtags, percentages, and time expressions.

## Exercise 3 — Token Offsets

Return token text, start position, and end position for ten examples.

## Exercise 4 — Sentence Segmentation

Protect abbreviations and decimal numbers before sentence splitting.

## Exercise 5 — Contractions

Compare three tokenization policies for English contractions.

## Exercise 6 — Arabic Segmentation

Define a documented policy for conjunctions, prepositions, definite articles, and
attached pronouns.

## Exercise 7 — Evaluation

Create reference tokenizations and calculate boundary precision, recall, and F1.

## Exercise 8 — Downstream Comparison

Compare two tokenizers in a classification experiment and analyze changed errors.

## Challenge Exercises

1. Build a multilingual tokenizer with language-specific rules.
2. Implement a sentence segmenter with abbreviation and decimal protection.
3. Create a tokenizer test suite containing at least fifty edge cases.
4. Compare word, character, and subword vocabulary sizes.
5. Package the tokenizer as a reusable Python module.

# 17. Summary and Next Lesson

In this lesson:

- tokenization divided text into task-specific processing units;
- whitespace splitting provided a simple but limited baseline;
- rule-based and regex tokenizers recognized structured expressions;
- punctuation, contractions, hyphens, numbers, URLs, and emails required explicit
  policies;
- offsets aligned tokens with original text;
- sentence segmentation required more than period splitting;
- abbreviations and decimals created boundary ambiguity;
- multilingual tokenization depended on writing system and morphology;
- Arabic tokenization required documented clitic-segmentation decisions;
- character and subword units reduced dependence on fixed word vocabularies;
- boundary metrics and downstream evaluation measured tokenizer quality.

## Next Lesson

**Lesson 13: Stemming and Lemmatization in Practice** applies normalization methods
to classical NLP pipelines and compares their effects on vocabulary, retrieval, and
classification.

# References

- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- Eisenstein, J. *Introduction to Natural Language Processing*.
- Python regular-expression documentation.
- Unicode text-segmentation documentation.
- NLTK and spaCy tokenization documentation.
- Arabic tokenization and morphological-segmentation literature.